In [1]:
# put this at the top of every notebook
import plotly.io as pio

#pio.renderers.default = "notebook_connected"
pio.renderers.default = "browser"   # always opens in browser, no nbformat needed


In [2]:
from yfinance_api3.classes.stock_client import StockClient
from yfinance_api3.classes.quant_analytics import QuantAnalytics
from yfinance_api3.classes.options import OptionsAnalyzer
from yfinance_api3.classes.positions_book import PositionsBook
import yfinance_api3.modules.plots as plots

client = StockClient()
quant  = QuantAnalytics(client)

In [3]:
# ── 1. Build the book — trade record only ─────────────────────────────────
book = PositionsBook("INTC")
book.add_option("call", "2026-12-18", 100, -60, 18.11)
book.add_option("call", "2026-12-18", 90, -40, 21.50)
book.add_underlying(10000, 30.0, "long")

In [4]:




# ── 2. Mark to market — fetch everything from market ──────────────────────
opt     = OptionsAnalyzer(client, "INTC")
updates = book.mark_to_market(client, opt=opt, quant=quant)
#print(f"Spot:      ${book.spot:,.2f}")
#print(f"Vol:       {book.vol:.1%}  (historical — for price range only)")
#print(f"Risk-free: {book.risk_free_rate:.2f}%  (^IRX)")
#print(f"Div yield: {book.div_yield:.2f}%")
#print(f"Leg IVs:   {updates.get('legs_iv')}")

# ── 3. Greeks and summary with live market data ───────────────────────────
#print(book.greeks_table().to_string())
print(book.summary())

# ── 4. Plots ──────────────────────────────────────────────────────────────
plots.positions_book(book, days_ahead=0,  d_std=3).show()
#plots.positions_book(book, days_ahead=30, d_std=3).show()
plots.positions_legs(book).show()

# ── 5. Save / reload ──────────────────────────────────────────────────────
book.save("positions/INTC.json")
book2 = PositionsBook.load("positions/INTC.json")
book2.mark_to_market(client, opt=OptionsAnalyzer(client, "INTC"), quant=quant)
print(book2)

{'symbol': 'INTC', 'spot': 99.62, 'vol': 0.6662, 'risk_free_rate': 3.575, 'div_yield': 0.0, 'days_ahead': 0, 'model_date': '2026-05-02', 'closed_pnl': 0.0, 'open_value': 431162.78, 'open_pnl': 540822.78, 'expected_pnl': 540822.78, 'delta': -3572.91, 'delta_dollars': -355933.3, 'gamma': -114.45, 'vega': -5658.01, 'theta': 1057.01, 'rho': -4949.87}
PositionsBook(INTC  spot=$99.62  legs=2  closed_pnl=$0.00)


In [5]:
# view full payoff table
curves = book.payoff_curves()
print(curves.to_string())

# or styled in notebook
curves.style.format("${:,.2f}", subset=curves.columns[1:])

# export to CSV
curves.to_csv("INTC_payoff.csv", index=False)

     underlyingValue  leg_6381bc63_call_90_hoy  leg_6381bc63_call_90_vcto  und_hoy  und_vcto    P&L Hoy   P&L Vcto
0              56.17                  61856.81                   86000.00      0.0       0.0   61856.81   86000.00
1              56.98                  60723.67                   86000.00      0.0       0.0   60723.67   86000.00
2              57.79                  59563.39                   86000.00      0.0       0.0   59563.39   86000.00
3              58.60                  58376.13                   86000.00      0.0       0.0   58376.13   86000.00
4              59.41                  57162.06                   86000.00      0.0       0.0   57162.06   86000.00
5              60.21                  55921.35                   86000.00      0.0       0.0   55921.35   86000.00
6              61.02                  54654.20                   86000.00      0.0       0.0   54654.20   86000.00
7              61.83                  53360.81                   86000.00      0

In [7]:
curves


,underlyingValue,leg_6381bc63_call_90_hoy,leg_6381bc63_call_90_vcto,und_hoy,und_vcto,P&L Hoy,P&L Vcto
0,56.17,61856.81,86000.00,0.0,0.0,61856.81,86000.00
1,56.98,60723.67,86000.00,0.0,0.0,60723.67,86000.00
2,57.79,59563.39,86000.00,0.0,0.0,59563.39,86000.00
3,58.60,58376.13,86000.00,0.0,0.0,58376.13,86000.00
4,59.41,57162.06,86000.00,0.0,0.0,57162.06,86000.00
...,...,...,...,...,...,...,...
145,173.45,-278324.52,-247781.80,0.0,0.0,-278324.52,-247781.80
146,174.25,-281307.52,-251016.99,0.0,0.0,-281307.52,-251016.99
147,175.06,-284294.18,-254252.17,0.0,0.0,-284294.18,-254252.17
148,175.87,-287284.44,-257487.36,0.0,0.0,-287284.44,-257487.36
